# Assignment 3: Sanskrit-English Sentence Embeddings using LaBSE

This notebook builds sentence-level Sanskrit-English embeddings using **LaBSE** with **PyTorch**, evaluates average cosine similarity on aligned pairs, and produces a **t-SNE visualization for 100 sample pairs**. The workflow follows the assignment constraints: only the provided dataset is used, pre-trained models are used locally, and no external APIs are called [file:1].

## Installation

Run this cell first if the environment does not already have the required packages.

In [ ]:
!pip -q install torch transformers sentence-transformers pandas scikit-learn matplotlib seaborn tqdm

## Imports and setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Dataset paths

Place the six CSV files in the same folder as this notebook, or update the paths below.

In [ ]:
TRAIN_SA_PATH = 'train_sa.csv'
TRAIN_EN_PATH = 'train_en.csv'
DEV_SA_PATH = 'dev_sa.csv'
DEV_EN_PATH = 'dev_en.csv'
TEST_SA_PATH = 'test_sa.csv'
TEST_EN_PATH = 'test_en.csv'

for path in [TRAIN_SA_PATH, TRAIN_EN_PATH, DEV_SA_PATH, DEV_EN_PATH, TEST_SA_PATH, TEST_EN_PATH]:
    print(path, '->', os.path.exists(path))

## Load and align parallel data

The Sanskrit files contain `Source_id` and `Sentence_sa`, and the English files contain `Source_id` and `Sentence_en`. The `Source_id` field aligns the sentence pairs [file:1].

In [ ]:
def load_parallel(sa_path, en_path):
    sa_df = pd.read_csv(sa_path)
    en_df = pd.read_csv(en_path)
    df = sa_df.merge(en_df, on='Source_id', how='inner')
    df = df[['Source_id', 'Sentence_sa', 'Sentence_en']].dropna().reset_index(drop=True)
    df['Sentence_sa'] = df['Sentence_sa'].astype(str).str.strip()
    df['Sentence_en'] = df['Sentence_en'].astype(str).str.strip()
    df = df[(df['Sentence_sa'] != '') & (df['Sentence_en'] != '')].reset_index(drop=True)
    return df

train_df = load_parallel(TRAIN_SA_PATH, TRAIN_EN_PATH)
dev_df = load_parallel(DEV_SA_PATH, DEV_EN_PATH)
test_df = load_parallel(TEST_SA_PATH, TEST_EN_PATH)

print('Train shape:', train_df.shape)
print('Dev shape:', dev_df.shape)
print('Test shape:', test_df.shape)
train_df.head()

## Load LaBSE

LaBSE is a multilingual sentence-embedding model designed for language-agnostic sentence representations, making it a strong fit for Sanskrit-English semantic similarity [web:11].

In [ ]:
MODEL_NAME = 'sentence-transformers/LaBSE'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print('Loaded model:', MODEL_NAME)

## Mean pooling and embedding generation

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

@torch.no_grad()
def encode_sentences(sentences, batch_size=32, max_length=128):
    all_embeddings = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        output = model(**encoded)
        embeddings = mean_pooling(output, encoded['attention_mask'])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()

## Generate embeddings

You can evaluate on `dev_df` first, and on `test_df` if labels are available for scoring in your local setup.

In [ ]:
dev_sa_emb = encode_sentences(dev_df['Sentence_sa'].tolist())
dev_en_emb = encode_sentences(dev_df['Sentence_en'].tolist())

embedding_dim = dev_sa_emb.shape[1]
print('Embedding dimension:', embedding_dim)

## Average cosine similarity

In [ ]:
pairwise_cos = np.sum(dev_sa_emb * dev_en_emb, axis=1)
avg_cosine_similarity = float(np.mean(pairwise_cos))

print('Embedding dimension used:', embedding_dim)
print('Average cosine similarity on dev pairs:', round(avg_cosine_similarity, 6))

## Optional: evaluate train and test splits

In [ ]:
def evaluate_split(df, split_name='split'):
    sa_emb = encode_sentences(df['Sentence_sa'].tolist())
    en_emb = encode_sentences(df['Sentence_en'].tolist())
    cos_scores = np.sum(sa_emb * en_emb, axis=1)
    avg_score = float(np.mean(cos_scores))
    print(f'{split_name} average cosine similarity: {avg_score:.6f}')
    return sa_emb, en_emb, avg_score

# Uncomment if needed
# train_sa_emb, train_en_emb, train_score = evaluate_split(train_df, 'Train')
# test_sa_emb, test_en_emb, test_score = evaluate_split(test_df, 'Test')

## t-SNE visualization for 100 sample pairs

The assignment explicitly asks for a 2D t-SNE projection of 100 Sanskrit-English pairs [file:1].

In [ ]:
sample_n = min(100, len(dev_df))
sample_df = dev_df.sample(sample_n, random_state=SEED).reset_index(drop=True)

sample_sa_emb = encode_sentences(sample_df['Sentence_sa'].tolist())
sample_en_emb = encode_sentences(sample_df['Sentence_en'].tolist())

combined_embeddings = np.vstack([sample_sa_emb, sample_en_emb])
labels = ['Sanskrit'] * sample_n + ['English'] * sample_n
pair_ids = list(range(sample_n)) + list(range(sample_n))

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init='pca', learning_rate='auto')
coords = tsne.fit_transform(combined_embeddings)

plot_df = pd.DataFrame({
    'x': coords[:, 0],
    'y': coords[:, 1],
    'Language': labels,
    'PairID': pair_ids
})

plt.figure(figsize=(12, 8))
sns.scatterplot(data=plot_df, x='x', y='y', hue='Language', style='Language', s=70, alpha=0.85)

for i in range(sample_n):
    plt.plot(
        [plot_df.iloc[i]['x'], plot_df.iloc[i + sample_n]['x']],
        [plot_df.iloc[i]['y'], plot_df.iloc[i + sample_n]['y']],
        color='gray', alpha=0.15, linewidth=0.8
    )

plt.title('t-SNE Projection of 100 Sanskrit-English Sentence Pairs (LaBSE)', fontsize=14)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(title='Language')
plt.grid(alpha=0.2)
plt.show()

## Save embeddings if required

In [ ]:
# Optional: save embeddings for further analysis
# np.save('dev_sa_labse.npy', dev_sa_emb)
# np.save('dev_en_labse.npy', dev_en_emb)

## Final reported outputs

This notebook prints the required outputs requested in the assignment: complete code, embedding dimensionality, average cosine similarity, and inline t-SNE visualization [file:1].

In [ ]:
print('Final model:', MODEL_NAME)
print('Final embedding dimension:', embedding_dim)
print('Final dev average cosine similarity:', round(avg_cosine_similarity, 6))